# Flammability index numbers

Same math as the atlas (`postfire_landscape_atlas1.html`).

## Time since fire

Perimeters: CAL FIRE FRAP, MTBS, burn-date / fire-occurrence records.

$$
\mathrm{TSF}_{\mathrm{days}} = \text{satellite date} - \text{last fire date}
$$

If several fires overlap a pixel, use the **most recent fire before** the Sentinel-2 date.
If there is **no recorded fire**, TSF = **30 years = 10,950 days**.

Example if the scene is 16 October 2026:

| Last fire | TSF (days) |
| --- | --- |
| 16 Oct 2025 | 365 |
| 16 Oct 2024 | 730 |
| 16 Oct 2020 | 2,191 |
| none | 10,950 |

## Flammability

$$
\mathrm{NPV}_{\mathrm{veg}} = \frac{\mathrm{NPV}}{\mathrm{NPV}+\mathrm{PV}}
\qquad
z = a\cdot\mathrm{TF} + b\cdot\mathrm{NPV}_{\mathrm{veg}} - c
$$

$$
\mathrm{FI}_A = \frac{1}{1+e^{-z}}
\qquad
\mathrm{FI}_B = e^{z}
$$

Defaults: $a=0.005$, $b=2$, $c=5$. TF is TSF in days.

Run on CyVerse next to the atlas files. Set `DATE` and `TILE` below, then run all cells.

In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta
import json, math, sys
import numpy as np
import pandas as pd

DATE = '2026-10-16'   # or '2026-07-28'
TILE = None           # None = first atlas scene; or folder name e.g. '11_S_LV_2026_10_16'

A, B, C = 0.005, 2.0, 5.0
NO_FIRE_YEARS = 30
NO_FIRE_DAYS = NO_FIRE_YEARS * 365  # 10,950

here = Path.cwd().resolve()
print('cwd', here)

In [ ]:
def first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None

def find_new_data():
    checks = []
    for parent in [here, *here.parents]:
        checks.append(parent / 'new_data')
        checks.append(parent / 'SCE-CU' / 'sentinel2_data' / 'new_data')
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
        Path('/home/jovyan/data-store/new_data'),
    ])
    found = first_existing(checks)
    if found is None:
        raise FileNotFoundError('Could not find new_data')
    return found

new_data = find_new_data()
atlas = new_data / DATE / 'atlas'
print('new_data', new_data)
print('atlas', atlas, 'exists' if atlas.exists() else 'MISSING')

## Formulas (same as the website)

In [ ]:
def parse_day(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, date):
        return val
    s = str(val).strip()[:10]
    for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%m/%d/%Y'):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None

def tsf_days(scene, last_fire):
    scene_d = parse_day(scene)
    fire_d = parse_day(last_fire)
    if scene_d is None:
        raise ValueError('Need a scene date')
    if fire_d is None or fire_d > scene_d:
        return NO_FIRE_DAYS
    return (scene_d - fire_d).days

def npv_veg(pv, npv):
    veg = float(pv) + float(npv)
    return 0.0 if veg < 1e-6 else float(npv) / veg

def z_linear(tf, pv, npv, a=A, b=B, c=C):
    return a * float(tf) + b * npv_veg(pv, npv) - c

def fi_a(tf, pv, npv, a=A, b=B, c=C):
    z = max(-60.0, min(60.0, z_linear(tf, pv, npv, a, b, c)))
    return 1.0 / (1.0 + math.exp(-z))

def fi_b(tf, pv, npv, a=A, b=B, c=C):
    z = max(-20.0, min(20.0, z_linear(tf, pv, npv, a, b, c)))
    return math.exp(z)

scene = date.fromisoformat(DATE)
examples = [
    ('A', date(2025, 10, 16), 0.12, 0.00),
    ('B', date(2024, 10, 16), 0.12, 0.00),
    ('C', date(2020, 10, 16), 0.12, 0.00),
    ('no fire', None, 0.12, 0.00),
]
rows = []
for name, last, pv, npv in examples:
    tf = tsf_days(scene, last)
    z = z_linear(tf, pv, npv)
    rows.append({
        'pixel': name,
        'last_fire': last.isoformat() if last else 'none',
        'TSF_days': tf,
        'NPV_veg': round(npv_veg(pv, npv), 4),
        'z': round(z, 3),
        'FI_A': round(fi_a(tf, pv, npv), 4),
        'FI_B': fi_b(tf, pv, npv),
    })
ex = pd.DataFrame(rows)
print(f'Scene date {scene}  |  a={A}, b={B}, c={C}  |  no fire = {NO_FIRE_DAYS} days')
display(ex)

## Load one atlas tile + fire perimeters

Uses `mesma_grid.js` (PV, NPV, …) and `atlas/tsf_years/fires.geojson` (or the Time_since_fire copy).

In [ ]:
def load_grid(js_path):
    raw = Path(js_path).read_text()
    marker = 'window.MESMA_GRID='
    payload = json.loads(raw.split(marker, 1)[1].strip().rstrip(';'))
    return payload

def find_fires():
    return first_existing([
        atlas / 'tsf_years' / 'fires.geojson',
        new_data / DATE / 'atlas' / 'tsf_years' / 'fires.geojson',
        new_data / 'Time_since_fire' / 'by_year_2015_2026' / 'fires.geojson',
        new_data / 'Time_since_fire' / 'by_year_2015_2020' / 'fires.geojson',
    ])

scenes = sorted(p for p in atlas.iterdir() if p.is_dir() and (p / 'mesma_grid.js').exists()) if atlas.exists() else []
if not scenes:
    raise FileNotFoundError(f'No mesma_grid.js under {atlas}')

if TILE:
    scene_dir = atlas / TILE
    if not (scene_dir / 'mesma_grid.js').exists():
        raise FileNotFoundError(scene_dir)
else:
    scene_dir = scenes[0]

print(f'{len(scenes)} atlas scenes. Using {scene_dir.name}')
G = load_grid(scene_dir / 'mesma_grid.js')
rows, cols = int(G['rows']), int(G['cols'])
scale = float(G.get('scale') or 1000)
ll = G['lonlat']
scene_date = parse_day(G.get('sceneDate') or DATE)
pv = np.array(G['bands']['pv'], dtype=np.int32).reshape(rows, cols)
npv = np.array(G['bands']['npv'], dtype=np.int32).reshape(rows, cols)
valid = pv >= 0
pv_f = np.where(valid, pv / scale, np.nan)
npv_f = np.where(valid, npv / scale, np.nan)
print('tile', G.get('tile'), 'sceneDate', scene_date, 'grid', rows, 'x', cols)
print('modeled cells', int(valid.sum()))

fires_path = find_fires()
print('fires', fires_path)
fires = json.loads(fires_path.read_text()) if fires_path else {'features': []}
print('fire features', len(fires.get('features') or []))

## TSF for every cell

Rasterize perimeters onto the atlas grid. Older fires first, newer last, so the last write is the most recent fire before the scene. Empty cells get 10,950 days.

In [ ]:
from rasterio.features import rasterize
from rasterio.transform import from_bounds

west, south, east, north = ll['west'], ll['south'], ll['east'], ll['north']
transform = from_bounds(west, south, east, north, cols, rows)

burned = []
for feat in fires.get('features') or []:
    props = feat.get('properties') or {}
    geom = feat.get('geometry')
    fire_d = parse_day(props.get('burn_date') or props.get('Ig_Date') or props.get('ALARM_DATE'))
    if geom is None or fire_d is None or fire_d > scene_date:
        continue
    days = (scene_date - fire_d).days
    burned.append((fire_d, days, geom))
burned.sort(key=lambda x: x[0])  # oldest first, newest overwrites

tsf = np.full((rows, cols), NO_FIRE_DAYS, dtype=np.int32)
if burned:
    shapes = [(geom, max(0, days) + 1) for _, days, geom in burned]
    painted = rasterize(
        shapes,
        out_shape=(rows, cols),
        transform=transform,
        fill=0,
        dtype='int32',
        all_touched=True,
    )
    hit = painted > 0
    tsf[hit] = painted[hit] - 1

tsf = np.where(valid, tsf, -1)
from_fire = valid & (tsf != NO_FIRE_DAYS) & (tsf >= 0)
assumed = valid & (tsf == NO_FIRE_DAYS)
print(f'fires used (on or before {scene_date}): {len(burned)}')
print(f'cells from a fire: {int(from_fire.sum()):,}')
print(f'cells no fire (30 yr / {NO_FIRE_DAYS} d): {int(assumed.sum()):,}')
print(f'mean TSF (modeled cells): {float(np.nanmean(np.where(valid, tsf, np.nan))):,.0f} d')

## FI-A and FI-B for the whole tile

These means are what the atlas dials are averaging.

In [ ]:
tf = tsf.astype(np.float64)
npv_v = np.divide(npv_f, pv_f + npv_f, out=np.zeros_like(pv_f, dtype=np.float64), where=(pv_f + npv_f) > 1e-6)
z = A * tf + B * npv_v - C
fiA = 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))
fiB = np.exp(np.clip(z, -20, 20))
fiA = np.where(valid, fiA, np.nan)
fiB = np.where(valid, fiB, np.nan)

summary = pd.DataFrame([
    {
        'scene': scene_dir.name,
        'scene_date': str(scene_date),
        'modeled_cells': int(valid.sum()),
        'from_fire': int(from_fire.sum()),
        'no_fire_30yr': int(assumed.sum()),
        'mean_TSF_days': float(np.nanmean(tf[valid])),
        'mean_NPV_veg': float(np.nanmean(npv_v[valid])),
        'mean_FI_A': float(np.nanmean(fiA)),
        'mean_FI_B': float(np.nanmean(fiB)),
        'a': A, 'b': B, 'c': C,
    }
])
display(summary.T.rename(columns={0: 'value'}))
print('FI-A (sigmoid 0-1) should match the FI-A dial (~0.99 when TSF is thousands of days).')
print('FI-B (exp) is unbounded and is clamped at exp(20) in the atlas.')

## One pixel

Set `ROW` and `COL` (atlas grid, not 10 m), or leave as the cell nearest the grid center.

In [ ]:
ROW = None
COL = None
if ROW is None or COL is None:
    rr, cc = np.where(valid)
    ROW, COL = int(rr[len(rr)//2]), int(cc[len(cc)//2])

if not valid[ROW, COL]:
    raise ValueError(f'r{ROW} c{COL} is not a modeled cell')

lat = ll['north'] - (ROW + 0.5) / rows * (ll['north'] - ll['south'])
lon = ll['west'] + (COL + 0.5) / cols * (ll['east'] - ll['west'])
tf1 = int(tsf[ROW, COL])
pv1 = float(pv_f[ROW, COL])
npv1 = float(npv_f[ROW, COL])
one = pd.DataFrame([
    {'term': 'location', 'value': f'{lat:.4f}, {lon:.4f}'},
    {'term': 'row, col', 'value': f'{ROW}, {COL}'},
    {'term': 'scene date', 'value': str(scene_date)},
    {'term': 'TSF days', 'value': tf1},
    {'term': 'TSF source', 'value': 'no fire · 30 years' if tf1 == NO_FIRE_DAYS else 'scene date − last fire date'},
    {'term': 'PV', 'value': round(pv1, 4)},
    {'term': 'NPV', 'value': round(npv1, 4)},
    {'term': 'NPV/(NPV+PV)', 'value': round(npv_veg(pv1, npv1), 4)},
    {'term': 'a·TF', 'value': round(A * tf1, 4)},
    {'term': 'b·NPV_veg', 'value': round(B * npv_veg(pv1, npv1), 4)},
    {'term': 'z = aTF + bNPV − c', 'value': round(z_linear(tf1, pv1, npv1), 4)},
    {'term': 'FI_A = 1/(1+e^{-z})', 'value': fi_a(tf1, pv1, npv1)},
    {'term': 'FI_B = e^{z}', 'value': fi_b(tf1, pv1, npv1)},
])
display(one)

## Optional: every atlas scene for this date

In [ ]:
RUN_ALL = False  # True = loop every tile for DATE (slower)

def summarize_dir(folder):
    g = load_grid(folder / 'mesma_grid.js')
    r, c = int(g['rows']), int(g['cols'])
    sc = float(g.get('scale') or 1000)
    lonlat = g['lonlat']
    sd = parse_day(g.get('sceneDate') or DATE)
    pvb = np.array(g['bands']['pv'], dtype=np.int32).reshape(r, c)
    npvb = np.array(g['bands']['npv'], dtype=np.int32).reshape(r, c)
    ok = pvb >= 0
    pvf = np.where(ok, pvb / sc, np.nan)
    npvf = np.where(ok, npvb / sc, np.nan)
    tr = from_bounds(lonlat['west'], lonlat['south'], lonlat['east'], lonlat['north'], c, r)
    t = np.full((r, c), NO_FIRE_DAYS, dtype=np.int32)
    shapes = []
    for feat in fires.get('features') or []:
        props = feat.get('properties') or {}
        geom = feat.get('geometry')
        fire_d = parse_day(props.get('burn_date') or props.get('Ig_Date') or props.get('ALARM_DATE'))
        if geom is None or fire_d is None or fire_d > sd:
            continue
        shapes.append((fire_d, (sd - fire_d).days, geom))
    shapes.sort(key=lambda x: x[0])
    if shapes:
        painted = rasterize(
            [(geom, max(0, days) + 1) for _, days, geom in shapes],
            out_shape=(r, c), transform=tr, fill=0, dtype='int32', all_touched=True,
        )
        hit = painted > 0
        t[hit] = painted[hit] - 1
    t = np.where(ok, t, -1)
    tfv = t.astype(np.float64)
    nv = np.divide(npvf, pvf + npvf, out=np.zeros_like(pvf, dtype=np.float64), where=(pvf + npvf) > 1e-6)
    zv = A * tfv + B * nv - C
    a = 1.0 / (1.0 + np.exp(-np.clip(zv, -60, 60)))
    b = np.exp(np.clip(zv, -20, 20))
    return {
        'scene': folder.name,
        'scene_date': str(sd),
        'modeled_cells': int(ok.sum()),
        'from_fire': int((ok & (t != NO_FIRE_DAYS) & (t >= 0)).sum()),
        'no_fire_30yr': int((ok & (t == NO_FIRE_DAYS)).sum()),
        'mean_TSF_days': float(np.nanmean(tfv[ok])),
        'mean_FI_A': float(np.nanmean(np.where(ok, a, np.nan))),
        'mean_FI_B': float(np.nanmean(np.where(ok, b, np.nan))),
    }

if RUN_ALL:
    table = pd.DataFrame([summarize_dir(p) for p in scenes])
    display(table)
    out = atlas / f'fi_summary_{DATE}.csv'
    table.to_csv(out, index=False)
    print('wrote', out)
else:
    print('Set RUN_ALL = True to score every tile for', DATE)